In [1]:
import torch

In [2]:
checkpoint = torch.load('/home/user/mkotovanu/neudc/assets/models/yolov8n.pt', map_location='cuda', weights_only=False)
if isinstance(checkpoint, dict):
    if "model" in checkpoint:
        model = checkpoint["model"]
        for key in ["metadata", "config", "meta"]:
            if key in checkpoint:
                metadata = checkpoint[key]
                break
        if "train_args" in checkpoint:
            metadata = checkpoint["train_args"]
    elif "state_dict" in checkpoint:
        raise ValueError("Model architecture required for state_dict loading")
    else:
        model = checkpoint
else:
    model = checkpoint

print("="*30, "\nMODEL GRAPH\n")
for i, m in enumerate(model.model):     # core.model == nn.Sequential([...])
    print(f"[{i:2d}] {m.__class__.__name__:<20}  out_ch={getattr(m, 'ch', '?')}")


MODEL GRAPH

[ 0] Conv                  out_ch=?
[ 1] Conv                  out_ch=?
[ 2] C2f                   out_ch=?
[ 3] Conv                  out_ch=?
[ 4] C2f                   out_ch=?
[ 5] Conv                  out_ch=?
[ 6] C2f                   out_ch=?
[ 7] Conv                  out_ch=?
[ 8] C2f                   out_ch=?
[ 9] SPPF                  out_ch=?
[10] Upsample              out_ch=?
[11] Concat                out_ch=?
[12] C2f                   out_ch=?
[13] Upsample              out_ch=?
[14] Concat                out_ch=?
[15] C2f                   out_ch=?
[16] Conv                  out_ch=?
[17] Concat                out_ch=?
[18] C2f                   out_ch=?
[19] Conv                  out_ch=?
[20] Concat                out_ch=?
[21] C2f                   out_ch=?
[22] Detect                out_ch=?


In [ ]:
def _hook(_, __, out):
    if out.ndim > 2:
        out = out.mean(dim=list(range(2, out.ndim)))  # GAP
    emb_holder["feat"] = out.detach()

core = getattr(model, "model", model)   # если .model нет – берём self.model
modules = list(core)[-2]
handle = modules.register_forward_hook(_hook)

TypeError: Module.register_forward_hook() missing 1 required positional argument: 'hook'

In [9]:
# %% [markdown]
# # Проверка: YOLOv8 .pt  ➜  эмбеддинги
# * модель грузим **torch.load**
# * эмбеддинг снимаем с предпоследнего модуля (по умолчанию –1 → Detect, -2 → последний C2f)

# %%
import cv2
import json
import math
import numpy as np
import torch
from pathlib import Path
from typing import List, Tuple

print("torch:", torch.__version__)

# %% [markdown]
# ## 1. Параметры

# %%
MODEL_PATH = Path("/home/user/mkotovanu/neudc/assets/models/yolov8n.pt")   # <-- укажите файл модели
IMG_DIR    = Path("/home/user/mkotovanu/neudc/assets/images")  # <-- папка с jpg/png
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE   = (960, 960)                    # входной размер, как при обучении
EMBED_LAYER_IDX = -2                      # -2 → последний C2f

# %% [markdown]
# ## 2. Загрузка модели

# %%
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE, weights_only=False)

if isinstance(checkpoint, dict) and "model" in checkpoint:
    model = checkpoint["model"]
    metadata = next(
        (checkpoint[k] for k in ("metadata", "config", "meta", "train_args") if k in checkpoint),
        {},
    )
else:
    model, metadata = checkpoint, {}

model = model.eval().to(DEVICE)
model = model.float()
print("Model type:", type(model))
print("Metadata keys:", list(metadata.keys())[:5])

# %% [markdown]
# ## 3. Вспомогательные функции

# %%
def letterbox(img: np.ndarray,
              new_shape: Tuple[int, int] = (640, 640),
              color=(114, 114, 114)
              ) -> np.ndarray:
    """Resize + pad to meet stride-size square."""
    h, w = img.shape[:2]
    new_h, new_w = new_shape
    scale = min(new_w / w, new_h / h)
    nh, nw = int(round(h * scale)), int(round(w * scale))

    pad_w, pad_h = new_w - nw, new_h - nh
    pad_w //= 2
    pad_h //= 2

    im = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    im = cv2.copyMakeBorder(im, pad_h, new_h - nh - pad_h,
                            pad_w, new_w - nw - pad_w,
                            cv2.BORDER_CONSTANT, value=color)
    return im

def preprocess(images: List[np.ndarray]) -> torch.Tensor:
    """BGR uint8 list → float32/640 tensor BCHW [0,1]"""
    ims = [letterbox(cv2.cvtColor(im, cv2.COLOR_BGR2RGB), IMG_SIZE) for im in images]
    ims = np.stack(ims).astype(np.float32) / 255.0            # B,H,W,C
    ims = torch.from_numpy(ims).permute(0, 3, 1, 2)           # B,C,H,W
    return ims.to(DEVICE)

# %% [markdown]
# ## 4. Функция извлечения эмбеддингов

# %%
def extract_embeddings(img_paths: List[Path],
                       embed_layer_idx: int = -2) -> np.ndarray:
    """
    Returns
    -------
    np.ndarray shape (B, C) – эмбеддинг каждой картинки
    """
    # 1. prepare batch
    ims = [cv2.imread(str(p)) for p in img_paths]
    x = preprocess(ims)

    # 2. choose target layer
    core = getattr(model, "model", model)        # DetectionModel или Sequential
    modules = list(core)
    target = modules[embed_layer_idx]

    # 3. register hook, run once
    holder = {}
    def hook(_, __, out):
        if out.ndim > 2:
            out = out.mean(dim=list(range(2, out.ndim)))  # GAP
        holder["feat"] = out.detach()

    h = target.register_forward_hook(hook)
    _ = model(x)          # forward; боксы не нужны
    h.remove()

    if "feat" not in holder:
        raise RuntimeError("hook failed – проверьте embed_layer_idx")

    return holder["feat"].cpu().numpy()

# %% [markdown]
# ## 5. Запуск

# %%
# берём первые N картинок из директории
img_list = sorted(IMG_DIR.glob("*.png"))[:16]
embeddings = extract_embeddings(img_list, embed_layer_idx=EMBED_LAYER_IDX)

print("Images:", len(img_list))
print("Embeddings shape:", embeddings.shape)  # (B, C)
for i in embeddings:
    print(i)


torch: 2.7.1+cu126
Model type: <class 'ultralytics.nn.tasks.DetectionModel'>
Metadata keys: ['task', 'mode', 'model', 'data', 'epochs']
Images: 1
Embeddings shape: (1, 256)
[    -0.0413    -0.20493    -0.17957    -0.16264    -0.21553     -0.1711   -0.092377    -0.10398    -0.10411    0.066302    -0.16577   -0.078315   -0.042748    -0.10501    -0.20972    -0.17236   -0.084898    -0.12858   -0.081004    -0.21739   -0.082924    -0.13164   -0.062582    -0.19116    -0.13336   -0.087936
    -0.20168    0.078115    -0.13398     -0.1322     0.17917    -0.12544   0.0034033   -0.031997   0.0098901    -0.21417    -0.14441   -0.081559    -0.19776     -0.1378    -0.20185    0.061476    -0.22016    -0.18031   0.0076562    0.094593   -0.089915   -0.054963     0.03658   -0.084752    -0.15291    -0.15238
    -0.05835   -0.053784    -0.18178   -0.093874   -0.059882    -0.12378     0.11856    -0.17106   -0.084299    -0.12442    -0.13005   -0.046995    -0.14964    0.078025     0.42161    -0.03509    -0.17